# Allee Effect Modeling
## Capstone Project Notebook

Imagine a small group of animals colonizing a new island, or a patch of rare plants reduced to just a few individuals. Common sense might suggest that with abundant resources and little competition, their population should grow rapidly. However, sometimes the opposite happens: small populations struggle to survive or reproduce, and their growth rate actually *increases* as the population becomes slightly larger or denser.

This counter-intuitive phenomenon is known as the **Allee Effect**, named after the ecologist Warder Clyde Allee who extensively studied social aggregation and cooperation in animals. It describes a situation in population ecology where, at *low* population sizes or densities, there is a **positive relationship** between the population size (or density) and the *per capita* (average individual) growth rate. In simpler terms, individuals in very small or sparse populations sometimes do worse in terms of survival and reproduction than individuals in slightly larger or denser populations.

This contrasts with the more familiar concept of density-dependent limitation (like in the logistic model), where growth rates *decrease* as population density increases due to factors like competition for resources, increased disease transmission, or predation. The Allee effect focuses specifically on the challenges faced at the *lower end* of the population size spectrum.

### In this notebook you will:
1. Understand the classical **logistic growth model** and its limitations
2. Learn what the **Allee effect** is and why it matters for conservation
3. Explore **weak** and **strong** Allee effect formulations
4. Implement the models step by step with stability analysis
5. Visualise population dynamics and compare model behaviours
6. Study a real-world ecological scenario
7. Build a foundation for the required project extensions

---

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('Imports loaded successfully.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('Imports loaded successfully.')

---
## 1 · Background: Logistic Growth

Before introducing the Allee effect, we need a baseline model to compare against. The classical logistic growth model describes a population $N(t)$ that grows towards a carrying capacity $K$:

$$\frac{dN}{dt} = r\,N\left(1 - \frac{N}{K}\right)$$

where:
- $r$ is the intrinsic growth rate — the maximum per-capita growth rate when the population is very small
- $K$ is the carrying capacity — the maximum sustainable population size given the available resources

**Key assumption:** the per-capita growth rate $f(N) = r(1 - N/K)$ is highest when $N$ is small and decreases linearly as $N \to K$.

This means: **any small population will grow.** But is that always true in nature?

The function below implements the right-hand side of this ODE. We then verify its key properties: growth should be zero at $N = 0$ (no individuals to reproduce) and at $N = K$ (population at capacity), and maximal at $N = K/2$.

In [ ]:
def logistic_rhs(N, r, K):
    """Right-hand side of the logistic growth equation."""
    return r * N * (1 - N / K)

print('Growth at the very start is zero:   {:.2f}'.format(logistic_rhs(0, 0.5, 100)))
print('Growth near the beginning is slow:   {:.2f}'.format(logistic_rhs(5, 0.5, 100)))
print('Growth at half capacity is the fastest:  {:.2f}'.format(logistic_rhs(50, 0.5, 100)))
print('Growth near the full capacity is slow:   {:.2f}'.format(logistic_rhs(95, 0.5, 100)))
print('Growth at the full capacity is zero: {:.2f}'.format(logistic_rhs(100, 0.5, 100)))
print('Growth at population above the capacity is negative: {:.2f}'.format(logistic_rhs(110, 0.5, 100)))

---
## 2 · The Allee Effect

The **Allee effect** (named after ecologist Warder Clyde Allee, 1931) describes a phenomenon where population growth rate is reduced or negative at **low population densities**.

### Why does this happen?

The Allee effect arises from mechanisms that require or benefit from a certain number of individuals being present. Key components contributing to the Allee effect include:

- **Mate finding difficulty** — individuals in sparse populations may struggle to find compatible mates. This is especially important for species with limited mobility or short breeding seasons.
- **Reduced cooperative defence** — small groups may be less effective at deterring predators compared to larger groups (e.g., colonial nesting birds mobbing predators).
- **Reduced cooperative feeding** — some species rely on group hunting or foraging strategies that become inefficient at low numbers (e.g., wolf packs, schooling fish).

<div class="alert alert-info"><b>Note:</b> These are not the only reasons for the Allee effect, but they are the most common ones. Try to find other reasons in your research project.</div>

Understanding the Allee effect is crucial because it implies that populations can have a **minimum viable size** or density. If a population drops below this threshold, its growth rate can become negative, leading it towards extinction even if resources are theoretically plentiful.

### Two types of Allee effects

Ecologists distinguish between two main types:

| Type | Definition | Consequence |
|------|-----------|-------------|
| **Strong Allee effect** | Characterised by a critical population threshold $A$. Below this threshold ($N < A$), the per capita growth rate becomes **negative**: $\frac{1}{N}\frac{dN}{dt} < 0$. | Population is destined to decline towards **extinction** unless boosted by immigration or other factors. |
| **Weak Allee effect** | The per capita growth rate is still positive at low densities, but it is **lower** than at slightly higher densities. There is no critical extinction threshold. | Population always grows, but more **slowly** when small compared to intermediate sizes. |

---
## 3 · Mathematical Models

The Allee effect can be incorporated into population growth models by modifying the per capita growth rate term. There is no single mathematical representation — several alternative formulations are used depending on whether the goal is to model strong vs. weak effects.

### Strong Allee Effect

The simplest model incorporating the strong Allee effect modifies the logistic equation:

$$\frac{dN}{dt} = r\,N\left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right)$$

where $A$ is the **Allee threshold** ($0 < A < K$).

**Equilibria** (setting $dN/dt = 0$):

1. $N^* = 0$ — extinction (**stable**)
2. $N^* = A$ — Allee threshold (**unstable** — separatrix)
3. $N^* = K$ — carrying capacity (**stable**)

The per capita growth rate is:

$$\frac{1}{N}\frac{dN}{dt} = r\left(\frac{N}{A} - 1\right)\left(1 - \frac{N}{K}\right)$$

which is negative at low and high densities, positive only in between.

### Weak Allee Effect

A simple way to model reduced growth at low population sizes *without* an extinction threshold:

$$\frac{dN}{dt} = \frac{r}{K}\,N^2\left(1 - \frac{N}{K}\right)$$

The $N^2$ term ensures slow growth when the population is small, but growth remains **always non-negative**. There is no critical threshold.

### Flexible (Boukal–Berec) Model

More general forms allow flexible interpolation between weak and strong Allee effects:

$$\frac{dN}{dt} = r\,N\left(1 - \frac{N}{K}\right)\left(\frac{N - A}{K}\right)$$

Here, the parameter $A$ determines the nature of the effect:
- $A < 0$: No Allee effect
- $A = 0$: Logistic growth
- $0 < A < K$: Strong Allee effect

These various formulations emphasise that the Allee effect is a **concept** rather than a single equation — different biological processes and assumptions lead to different mathematical expressions.

### Step 1: Implement the Allee effect right-hand side

We now translate the strong Allee effect equation into Python. The function `allee_rhs` computes $dN/dt = r\,N\,(1 - N/K)\,(N/A - 1)$ for given population size, growth rate, carrying capacity, and Allee threshold.

After defining it, we verify the three equilibria analytically predicted above: $dN/dt$ must be exactly zero at $N = 0$, $N = A$, and $N = K$. We also check the sign of $dN/dt$ below and above the threshold to confirm the qualitative behaviour — negative growth below $A$, positive growth between $A$ and $K$.

In [ ]:
def allee_rhs(N, r, K, A):
    """Right-hand side of the strong Allee effect model.
    
    Parameters
    ----------
    N : float or array  — current population
    r : float           — intrinsic growth rate
    K : float           — carrying capacity
    A : float           — Allee threshold (0 < A < K)
    
    Returns
    -------
    dN/dt : float or array
    """
    return r * N * (1 - N / K) * (N / A - 1)

r, K, A = 1.0, 100.0, 20.0
print('Growth at extinction is zero: {:.2f}'.format(allee_rhs(0, r, K, A)))
print('Growth below threshold is negative: {:.2f}'.format(allee_rhs(10, r, K, A)))
print('Growth at the Allee threshold is zero: {:.2f}'.format(allee_rhs(A, r, K, A)))
print('Growth between A and K is positive: {:.2f}'.format(allee_rhs(50, r, K, A)))
print('Growth at carrying capacity is zero: {:.2f}'.format(allee_rhs(K, r, K, A)))
print('Growth above carrying capacity is negative: {:.2f}'.format(allee_rhs(110, r, K, A)))

### Comparing all three growth models

To build intuition for the differences between the logistic, weak Allee, and strong Allee models, we plot them side by side. The left panel shows the total growth rate $dN/dt$ as a function of population size — this tells us *how fast* the population changes. The right panel shows the *per-capita* growth rate $(1/N)\,dN/dt$ — this tells us how well each *individual* is doing on average.

We also define `weak_allee_rhs`, implementing the weak Allee model $dN/dt = (r/K)\,N^2(1 - N/K)$. Notice how the $N^2$ term suppresses growth at low densities without ever making it negative.

<div class="alert alert-info"><b>Note:</b> Make sure that you understand the difference between total and per-capita growth rate and can interpret both plots.</div>


In [ ]:
def weak_allee_rhs(N, r, K):
    """Right-hand side of the weak Allee effect model: (r/K)*N^2*(1 - N/K)."""
    return (r / K) * N**2 * (1 - N / K)

N_range = np.linspace(0, 120, 500)
r, K, A_strong = 1.0, 100.0, 20.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: dN/dt
ax1.plot(N_range, logistic_rhs(N_range, r, K), 'b-', lw=2, label='Logistic (no Allee)')
ax1.plot(N_range, weak_allee_rhs(N_range, r, K), 'orange', lw=2, label='Weak Allee')
ax1.plot(N_range, allee_rhs(N_range, r, K, A_strong), 'r-', lw=2, label=f'Strong Allee (A={A_strong})')
ax1.axhline(0, color='gray', lw=0.8, ls='--')
ax1.set_xlabel('Population $N$'); ax1.set_ylabel('$dN/dt$')
ax1.set_title('Population Growth Rate', fontweight='bold')
ax1.legend(fontsize=9)

# Right: per-capita growth rate
N_pos = N_range[N_range > 0.5]
ax2.plot(N_pos, logistic_rhs(N_pos, r, K) / N_pos, 'b-', lw=2, label='Logistic')
ax2.plot(N_pos, weak_allee_rhs(N_pos, r, K) / N_pos, 'orange', lw=2, label='Weak Allee')
ax2.plot(N_pos, allee_rhs(N_pos, r, K, A_strong) / N_pos, 'r-', lw=2, label='Strong Allee')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_xlabel('Population $N$'); ax2.set_ylabel('Per-capita growth rate')
ax2.set_title('Per-Capita Growth Rate', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

### Step 2: Visualise the growth rate function

Plotting $dN/dt$ vs $N$ is the single most informative diagnostic for any one-dimensional ODE model. Where the curve is above zero, the population grows; where it is below zero, the population declines. The points where the curve crosses zero are the equilibria, and the slope at those crossings tells us about stability.

**Left panel:** In the Allee model, $dN/dt$ is negative for $N < A$ — populations below the threshold decline.

**Right panel:** The per-capita growth rate dips below zero at low $N$ — this is the signature of the Allee effect.

In [ ]:
N_range = np.linspace(0, 120, 500)
r, K, A = 1.0, 100.0, 20.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: dN/dt vs N
dN_logistic = logistic_rhs(N_range, r, K)
dN_allee = allee_rhs(N_range, r, K, A)

ax1.plot(N_range, dN_logistic, 'b-', lw=2, label='Logistic')
ax1.plot(N_range, dN_allee, 'r-', lw=2, label=f'Allee (A={A})')
ax1.axhline(0, color='gray', lw=0.8, ls='--')
ax1.axvline(A, color='red', lw=0.8, ls=':', alpha=0.5, label=f'Threshold A={A}')
ax1.set_xlabel('Population $N$')
ax1.set_ylabel('Growth rate $dN/dt$')
ax1.set_title('Growth Rate: Logistic vs Allee Effect', fontweight='bold')
ax1.legend()

# Right: per-capita growth rate
N_pos = N_range[N_range > 0.5]  # avoid division by zero
pc_logistic = logistic_rhs(N_pos, r, K) / N_pos
pc_allee = allee_rhs(N_pos, r, K, A) / N_pos

ax2.plot(N_pos, pc_logistic, 'b-', lw=2, label='Logistic')
ax2.plot(N_pos, pc_allee, 'r-', lw=2, label=f'Allee (A={A})')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.axvline(A, color='red', lw=0.8, ls=':', alpha=0.5)
ax2.set_xlabel('Population $N$')
ax2.set_ylabel('Per-capita growth rate $(1/N)\, dN/dt$')
ax2.set_title('Per-Capita Growth Rate', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 4 · Numerical Simulation

To study how populations evolve over time, we solve the ODE numerically using `scipy.integrate.solve_ivp`, a robust adaptive solver that automatically adjusts the time step for accuracy. This is more reliable than manual Euler integration and is the standard approach for ODE systems.

In [ ]:
def simulate_allee(N0, r, K, A, T):
    """Simulate the strong Allee effect model using solve_ivp.
    
    Returns
    -------
    t : array of time points
    N : array of population values
    """
    def ode(t, N):
        return [allee_rhs(N[0], r, K, A)]
    
    sol = solve_ivp(ode, [0, T], [N0], dense_output=True, max_step=0.1)
    t = np.linspace(0, T, 500)
    N = sol.sol(t)[0]
    N = np.maximum(N, 0)  # prevent negative population
    return t, N

### Visualise the dynamics

We now run the simulation for a range of initial conditions to see the Allee threshold in action. Each curve starts at a different $N_0$ and evolves under the same parameters ($r = 1$, $K = 100$, $A = 20$). The resulting "fan" of trajectories clearly separates into two families: those that converge to $K$ (starting above $A$) and those that collapse to zero (starting below $A$). The boundary between the two families lies exactly at the unstable equilibrium $N^* = A$.

**Key observation:** Populations starting above $A = 20$ grow to $K = 100$. Populations starting below $A = 20$ decline to extinction. The Allee threshold $A$ is an **unstable equilibrium** — a tipping point.

In [ ]:
r, K, A = 1.0, 100.0, 20.0
initial_conditions = [5, 15, 19, 21, 25, 50, 80, 110]

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.viridis(np.linspace(0, 1, len(initial_conditions)))

for N0, c in zip(initial_conditions, cmap):
    t, N = simulate_allee(N0, r, K, A, T=15)
    ax.plot(t, N, color=c, lw=2, label=f'$N_0 = {N0}$')

ax.axhline(A, color='red', ls='--', lw=1.5, alpha=0.7, label=f'Allee threshold $A = {A}$')
ax.axhline(K, color='green', ls='--', lw=1.5, alpha=0.7, label=f'Carrying capacity $K = {K}$')
ax.set_xlabel('Time')
ax.set_ylabel('Population $N$')
ax.set_title('Strong Allee Effect: Population Dynamics', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_ylim(-5, 120)
plt.tight_layout()
plt.show()

---
## 5 · Comparing Logistic and Allee Models

The most striking consequence of the Allee effect becomes apparent when we compare both models side by side with the same initial conditions. In the standard logistic model, *any* positive initial population — no matter how small — eventually grows to the carrying capacity $K$. In the Allee model, populations starting below the threshold $A$ are doomed to extinction.

Below we compare the logistic and Allee models for small ($N_0 = 5, 15$) and larger ($N_0 = 30, 50$) starting populations.

**Left panel:** Small populations survive in the logistic model but go extinct with the Allee effect.

**Right panel:** Above the threshold, both models converge to $K$ (but the Allee model is initially slower because its growth rate is reduced near $A$).

In [ ]:
def simulate_logistic(N0, r, K, T):
    """Simulate the standard logistic growth model using solve_ivp."""
    def ode(t, N):
        return [logistic_rhs(N[0], r, K)]
    
    sol = solve_ivp(ode, [0, T], [N0], dense_output=True, max_step=0.1)
    t = np.linspace(0, T, 500)
    N = sol.sol(t)[0]
    N = np.maximum(N, 0)
    return t, N

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Case: N0 = 5 (below Allee threshold)
for N0 in [5, 15]:
    t_log, N_log = simulate_logistic(N0, r=1.0, K=100, T=15)
    t_all, N_all = simulate_allee(N0, r=1.0, K=100, A=20, T=15)
    ax1.plot(t_log, N_log, 'b-', lw=2, alpha=0.7)
    ax1.plot(t_all, N_all, 'r--', lw=2, alpha=0.7)
ax1.set_title(f'Small initial populations ($N_0 = 5, 15$)', fontweight='bold')
ax1.set_xlabel('Time')
ax1.set_ylabel('Population')
ax1.legend(['Logistic', 'Allee'], fontsize=10)

# Case: N0 = 50 (above Allee threshold)
for N0 in [30, 50]:
    t_log, N_log = simulate_logistic(N0, r=1.0, K=100, T=15)
    t_all, N_all = simulate_allee(N0, r=1.0, K=100, A=20, T=15)
    ax2.plot(t_log, N_log, 'b-', lw=2, alpha=0.7)
    ax2.plot(t_all, N_all, 'r--', lw=2, alpha=0.7)
ax2.set_title(f'Larger initial populations ($N_0 = 30, 50$)', fontweight='bold')
ax2.set_xlabel('Time')
ax2.set_ylabel('Population')
ax2.legend(['Logistic', 'Allee'], fontsize=10)

plt.suptitle('Logistic vs Strong Allee Effect', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6 · Effect of the Allee Threshold

How does the value of $A$ change the dynamics? The Allee threshold is a species-specific property determined by the underlying biological mechanisms (mate-finding efficiency, group defence strength, etc.). A species with a high $A$ is inherently more vulnerable — it needs a larger population just to sustain itself.

Below we fix the initial population at $N_0 = 25$ and sweep the threshold $A$ through several values. When $A < N_0$, the population is above the threshold and grows to $K$. When $A > N_0$, the population is below the threshold and collapses. The transition is sharp — a small increase in $A$ can flip the outcome from survival to extinction.

**Key insight:** With $N_0 = 25$, the population survives when $A < 25$ and goes extinct when $A > 25$. The Allee threshold is a **critical tipping point** for species persistence.

In [ ]:
N0 = 25
r, K = 1.0, 100.0
A_values = [10, 20, 25, 30, 40]

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.coolwarm(np.linspace(0, 1, len(A_values)))

for A, c in zip(A_values, cmap):
    t, N = simulate_allee(N0, r, K, A, T=20)
    ax.plot(t, N, color=c, lw=2, label=f'$A = {A}$')

ax.axhline(N0, color='gray', ls=':', lw=1, alpha=0.5, label=f'Initial $N_0 = {N0}$')
ax.set_xlabel('Time')
ax.set_ylabel('Population $N$')
ax.set_title(f'Effect of Allee Threshold (fixed $N_0 = {N0}$)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7 · Effect of the Allee Threshold

How does the value of $A$ change the dynamics? The Allee threshold is a species-specific property determined by the underlying biological mechanisms (mate-finding efficiency, group defence strength, etc.). A species with a high $A$ is inherently more vulnerable — it needs a larger population just to sustain itself.

Below we fix the initial population at $N_0 = 25$ and sweep the threshold $A$ through several values. When $A < N_0$, the population is above the threshold and grows to $K$. When $A > N_0$, the population is below the threshold and collapses. The transition is sharp — a small increase in $A$ can flip the outcome from survival to extinction.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

N0 = 25  # fixed initial population
thresholds = [15, 20, 25, 30, 35]
colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(thresholds)))

# Left: Time series for different thresholds
for A_val, c in zip(thresholds, colors):
    t, N = simulate_allee(N0, r=1.0, K=100, A=A_val, T=15)
    if abs(N[-1] - N0) < 1.0:
        fate = 'stationary'
    elif N[-1] > 50:
        fate = 'grows'
    else:
        fate = 'extinct'
    ax1.plot(t, N, color=c, lw=2, label=f'$A = {A_val}$ ({fate})')

ax1.axhline(N0, color='gray', ls=':', lw=1, label=f'$N_0 = {N0}$')
ax1.set_xlabel('Time')
ax1.set_ylabel('Population $N$')
ax1.set_title(f'Effect of Allee Threshold ($N_0 = {N0}$, $K = 100$)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(-5, 110)

# Right: Vector field for each threshold value
N_points = np.linspace(0, 120, 30)
for A_val, c in zip(thresholds, colors):
    dN_points = allee_rhs(N_points, r=1.0, K=100, A=A_val)
    # Plot as a line showing growth rate
    ax2.plot(N_points, dN_points, color=c, lw=2, alpha=0.7, label=f'$A = {A_val}$')

ax2.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
ax2.axvline(N0, color='gray', ls=':', lw=1.5, alpha=0.7, label=f'$N_0 = {N0}$')
ax2.set_xlabel('Population $N$')
ax2.set_ylabel('Growth rate $dN/dt$')
ax2.set_title('Growth Rate Functions', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlim(-5, 125)

plt.tight_layout()
plt.show()

---
## 8 · Phase Line Analysis

A **phase line** (or phase portrait for a 1D system) is the definitive qualitative tool for understanding ODE dynamics. We plot $dN/dt$ on the vertical axis against $N$ on the horizontal axis. The key features are:

- **Zero crossings** — these are the equilibria $N^*$ where the population is stationary.
- **Sign of $dN/dt$** — where the curve is positive (green region), the population increases; where negative (red region), it decreases.
- **Stability** — a stable equilibrium has the curve crossing from positive to negative (population is "attracted" from both sides). An unstable equilibrium has the opposite crossing (population is "repelled").

For the strong Allee model, the phase line reveals the three equilibria at a glance: two stable attractors ($N^* = 0$ and $N^* = K$) separated by the unstable threshold ($N^* = A$). This makes the "tipping point" nature of $A$ visually obvious.

In [ ]:
N_range = np.linspace(0, 120, 500)
r, K, A = 1.0, 100.0, 20.0
dN = allee_rhs(N_range, r, K, A)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(N_range, dN, 'k-', lw=2)
ax.fill_between(N_range, dN, 0, where=dN > 0, alpha=0.2, color='green', label='Growth')
ax.fill_between(N_range, dN, 0, where=dN < 0, alpha=0.2, color='red', label='Decline')
ax.axhline(0, color='gray', lw=0.8)

# Mark equilibria
equil = [(0, 'stable'), (A, 'unstable'), (K, 'stable')]
for N_eq, stability in equil:
    marker = 'o' if stability == 'stable' else 'x'
    color = 'green' if stability == 'stable' else 'red'
    ax.plot(N_eq, 0, marker, color=color, markersize=12, markeredgewidth=3, zorder=5)
    ax.annotate(f'$N^* = {N_eq}$\n({stability})', xy=(N_eq, 0),
                xytext=(N_eq, max(dN)*0.3), fontsize=10, ha='center',
                arrowprops=dict(arrowstyle='->', color=color))

ax.set_xlabel('Population $N$')
ax.set_ylabel('$dN/dt$')
ax.set_title('Phase Line Analysis: Strong Allee Effect', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9 · Why is the Allee Effect Important?

Recognising and understanding the Allee effect has significant implications across several fields:

- **Conservation Biology**: It helps explain why small, fragmented populations are particularly vulnerable to extinction and highlights the need to maintain populations above critical thresholds. Recovery plans may need to focus on increasing population size or density directly.
- **Invasion Biology**: It can explain why some introduced species fail to establish despite suitable conditions — they may arrive in numbers too small to overcome Allee effects (e.g., finding mates).
- **Pest Management**: For pests exhibiting Allee effects, control strategies that drive the population below the critical threshold $A$ can be particularly effective, leading to eradication rather than just suppression.
- **Resource Management**: In fisheries or wildlife harvesting, Allee effects mean that over-harvesting can push a population below a recovery threshold, making sustainable management more complex.
- **Ecological Theory**: It adds crucial realism to population models, demonstrating that growth is not always maximal at the lowest densities.

### Example Scenario: Colonial Seabirds

Consider a species of colonial seabird that nests in dense groups for protection against gulls:

- **Very low density (below $A$):** If only a few pairs try to nest, they are easily spotted and preyed upon by gulls. Nesting success (offspring produced per pair) is very low, potentially zero or negative net growth. Mate finding might also be harder if individuals are spread out. This population is likely to decline.
- **Intermediate density (between $A$ and $K$):** As more pairs join the colony, they benefit from communal defence ("safety in numbers"). More eyes spot predators, and collective mobbing can drive gulls away. Nesting success increases significantly. The population grows.
- **High density (approaching $K$):** As the colony becomes very crowded, competition for the best nesting sites increases, disease might spread more easily, and local food resources might become depleted. Nesting success starts to decline again. Growth slows and eventually stops.

This scenario illustrates how the per capita success rate (and thus growth rate) can be lowest at very low densities (the Allee effect) before declining again at very high densities (resource limitation).

---
## 10 · Your Tasks

The code above provides the baseline implementation. For your capstone project, you must extend it by implementing **at least two** of the following features.

### Task A: Allee Threshold Variability
Introduce different threshold values for the Allee effect and analyse how they impact population persistence or extinction.

- Sweep $A$ over a range (e.g., 5 to 45) and for each value, determine the minimum $N_0$ that leads to survival.
- Create a **bifurcation diagram**: plot the final population as a function of $A$ for a fixed $N_0$.
- Discuss the ecological meaning: what does a high $A$ imply about a species?

### Task B: Stochastic Effects
Implement random fluctuations in the growth rate or add demographic noise.

A simple approach: add a noise term to the Euler step:

$$N_{t+1} = N_t + \Delta t \cdot f(N_t) + \sigma \sqrt{N_t} \cdot \xi_t$$

where $\xi_t \sim \mathcal{N}(0, \Delta t)$ and $\sigma$ controls noise intensity.

- Run many realisations (e.g., 100) with the same $N_0$ slightly above $A$.
- Compute the **extinction probability** as a function of $N_0$.
- Show that stochasticity makes the effective threshold **larger** than $A$.

### Task C: Real-World Data Comparison
Compare your model's behaviour to real-world population data or existing ecological studies.

- Look for species where the Allee effect has been documented (e.g., Atlantic cod, African wild dog, passenger pigeon).
- Fit the model to the data using `scipy.optimize.curve_fit`.
- Discuss whether the strong Allee model is appropriate.

### Task D: Custom Feature
Another feature of similar complexity of your choice. Ideas:
- **Harvesting model**: add a constant or proportional harvest term $-h$ or $-hN$ and find the maximum sustainable yield.
- **Two-patch model**: two coupled populations with migration, where one may rescue the other from extinction.
- **Allen-Cahn framework**: implement the approach from the suggested journal club paper (see below).

---
## 11 · Discussion Points

In your final report, be sure to address:

1. **Implementation**: Explain how you implemented the logistic growth model and incorporated the Allee effect mathematically.
2. **Visualisations**: Demonstrate your results using visual plots and graphs.
3. **Additional Features**: Present the additional features you implemented, discussing your approach.
4. **Insights & Implications**: Discuss insights gained from modeling the Allee effect and its implications for conservation biology and population management.
5. **Limitations**: What assumptions does this model make? When might they fail?

---

## Recommended Reading & Journal Club

### Basic References

**1. Drake, J. M. & Kramer, A. M. (2011)**
*Allee effects.*
Nature Education Knowledge, 3(10), 2.
[Link](https://www.nature.com/scitable/knowledge/library/allee-effects-19699394)
→ Accessible introduction to Allee effects for a broad audience.

**2. Kramer, A. M., Berec, L. & Drake, J. M. (2017)**
*Editorial: Allee effects in ecology and evolution.*
Journal of Animal Ecology, 87(1), 7–10. [DOI](https://doi.org/10.1111/1365-2656.12777)
→ Editorial overview of current research directions on Allee effects.

**3. Wikipedia (2025)**
*Allee effect.*
[Link](https://en.wikipedia.org/w/index.php?title=Allee_effect&oldid=1274257082)
→ General overview of the concept, terminology, and biological examples.

---

### Journal Club Papers

**4. Mumby, P. J. et al. (2024)**
*Allee effects limit coral fertilization success.*
Proceedings of the National Academy of Sciences, 121(52). [DOI](https://doi.org/10.1073/pnas.2418314121)
→ Empirical demonstration of Allee effects in coral reproduction — shows how low population density reduces fertilization success in broadcast spawners.

**5. Fadai, N. T., Johnston, S. T. & Simpson, M. J. (2020)**
*Unpacking the Allee effect: determining individual-level mechanisms that drive global population dynamics.*
Proceedings of the Royal Society A, 476(2241). [DOI](https://doi.org/10.1098/rspa.2020.0350)
→ Connects individual-level mechanisms (mate finding, cooperative defence) to population-level Allee dynamics using mathematical models.

**6. Tobin, P. C., Berec, L. & Liebhold, A. M. (2011)**
*Exploiting Allee effects for managing biological invasions.*
Ecology Letters, 14(6), 615–624. [DOI](https://doi.org/10.1111/j.1461-0248.2011.01614.x)
→ Practical application of Allee effect theory to invasive species management — shows how pushing populations below the Allee threshold can achieve eradication.